
### Install Duck Duck Go and Trafilatura (scrapping tool) libraries
##### (another scrapping tool is 'Beautiful Soup')

#### also AgentAI is also known as ReAct (Reasoning and acting) Refer the link below
https://www.ibm.com/think/topics/react-agent

### Eval Podcast -  https://www.youtube.com/watch?v=BsWxPI9UM4c
### Anthropic document explaining agents vs workflow - https://www.anthropic.com/engineering/building-effective-agents

In [ ]:
# one time installation; commented after installation
# !pip install ddgs trafilatura
# !pip install openai-agents

### step1: Setup

In [1]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from IPython.display import Markdown, display
from ddgs import DDGS
from agents import Agent, Runner, function_tool
import trafilatura


load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception ("API key is missing.")
else:
    print(OPENAI_API_KEY[:8])


MODEL = "gpt-4.1-mini"

sk-proj-


### Step 2 : Define the tools

In [2]:
@function_tool
def search_web(query: str):
    """ Search the web using DubkDuckGo browser. Reeturn 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f" \u2705 Got results")
    return json.dumps(results, indent=2) # this step is to covert list into text as LLM underestands only text and not the list

In [3]:
@function_tool
def fetch_url(url:str):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 Got text: {len(text)} chars")
            return text
    print(f" \u274C failed to fech or extract text from url: {url}")
    return f" Could not extract text from {url}. Try a different source"




### Steep 3: The System Prompt
### This tells the LLM who it is and how to behave. The key things:

### 1) What it job is?
### 2) What tools it has?
### 3) What process to follow?
### 4) what output format to product?

In [4]:

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather informaton from at least 6 distinct sources before delviering your brief.
if you have a fewer than 6 sources then keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step3 : Define Agent

In [5]:
research_agent = Agent(
    name = "Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools = [search_web, fetch_url]
)

### Lets Run it

In [ ]:
result = await Runner.run(
    research_agent,
    input = "Research a following topic and provide a comprehensive result: how is AI used in health industries",
    max_turns=30
)

 ✅ Got results
 ✅ Got text: 25023 chars
 ✅ Got text: 99483 chars
 ✅ Got text: 5430 chars
 ✅ Got results
 ✅ Got text: 20290 chars
 ✅ Got text: 25850 chars
 ✅ Got results
 ✅ Got results
 ✅ Got results
 ❌ failed to fech or extract text from url: https://themedicalpractice.com/technology/innovation/future-of-ai-in-healthcare/
 ✅ Got text: 17235 chars
 ✅ Got text: 36169 chars
 ✅ Got text: 6310 chars


In [ ]:
print(f"Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))